# pandas 之十四：批量合并多个文件

---

这是**日常最实用**的一篇。

真实工作里，数据经常被拆成很多个文件：
每个月的销售表、每个地区的报表、每个人交上来的 Excel……
结构完全一样，但散落在几十个文件里。手工复制粘贴？不可能。

这一篇就用"**全国销售数据**"这个场景，把批量合并讲透，
顺带踩一个**几乎所有人都会踩的坑**：结果文件被当成输入，数据越跑越多。

## 学习目标

1. 用 `os.listdir` / `pathlib.glob` 遍历目录，筛选出目标文件
2. 用 `pd.concat` 把多个 DataFrame 摞成一张表
3. 给合并结果加一列**来源标识**，方便回溯
4. 明白"**输出写到输入目录**"为什么会造成数据翻倍（以及怎么避免）
5. 写出健壮、可重复运行的合并脚本


In [ ]:
import os
import glob
import numpy as np
import pandas as pd

pd.set_option('display.unicode.east_asian_width', True)
pd.set_option('display.width', 180)
print('pandas 版本：', pd.__version__)

# 所有演示文件都放在 生成文件/ 下面，不往项目根目录乱丢
BASE_DIR = os.path.join(os.getcwd(), '生成文件')
INPUT_DIR = os.path.join(BASE_DIR, '全国销售数据')       # 待合并的原始报表
MERGED_DIR = os.path.join(BASE_DIR, '合并结果')          # 合并结果单独放

print('输入目录：', INPUT_DIR)
print('输出目录：', MERGED_DIR)


---
# 一、先造一批"结构相同的 Excel"

现实中这些文件是别人给你的。为了能跑通，我们先**用代码造出来**——
这也顺便告诉你：**演示数据最好自己生成**，这样别人拿到你的代码也能跑。


In [ ]:
# 建好目录即可 —— 这里**一个文件都不删**。
#
# 【为什么不先"清空目录"？】
# 1. 清空目录要看"删除权限"：有些环境（同步盘、受管控的机器、
#    带安全策略的执行环境）会把批量删除直接拦下来，程序就卡在这里了；
# 2. 更重要的是——**下面写入的是固定的 4 个文件名**，
#    直接覆盖就能保证每次结果一致，**根本不需要先删**。
#
# 一句话：**能用"覆盖"解决的，就别用"删除"。**
for d in (INPUT_DIR, MERGED_DIR):
    os.makedirs(d, exist_ok=True)          # 不存在就建，存在也不报错

# 四个地区的销售报表，字段结构完全一致
np.random.seed(42)      # 固定随机种子，保证每次生成的数据一样

regions = ['华东', '华北', '华南', '西南']
products = ['手机', '电脑', '平板', '耳机']

for region in regions:
    n = np.random.randint(3, 6)          # 每个地区 3~5 条
    one = pd.DataFrame({
        '日期': pd.date_range('2024-01-01', periods=n, freq='D'),
        '商品': np.random.choice(products, n),
        '数量': np.random.randint(1, 20, n),
        '单价': np.random.choice([199, 999, 2999, 499], n),
    })
    path = os.path.join(INPUT_DIR, f'{region}.xlsx')
    one.to_excel(path, index=False)
    print(f'生成 {region}.xlsx，{len(one)} 行')

print('\n目录内容：')
for f in sorted(os.listdir(INPUT_DIR)):
    print('  ', f)


In [ ]:
# 看看其中一个长什么样
print('华东.xlsx 的内容：')
print(pd.read_excel(os.path.join(INPUT_DIR, '华东.xlsx')))


---
# 二、最直接的写法：遍历目录 + `concat`

核心就三步：

1. **列出文件** → `os.listdir()` 或 `glob.glob()`
2. **逐个读进来** → `pd.read_excel()`
3. **摞在一起** → `pd.concat([...])`

```python
pd.concat(
    objs,               # 要拼接的 DataFrame 列表
    axis=0,             # 0 = 上下摞（默认），1 = 左右拼
    ignore_index=True,  # 忽略原来的行号，重新编号
    join='outer',       # 列名不一致时：outer 取并集并补 NaN
    keys=None,          # 给每块加一层来源标识（也可以自己加列）
)
```


In [ ]:
df = pd.DataFrame()      # 从空表开始（原稿的写法）

for file in sorted(os.listdir(INPUT_DIR)):        # 【改进】原稿没排序，文件顺序不确定
    file_path = os.path.join(INPUT_DIR, file)

    if file.endswith('.xlsx'):
        one = pd.read_excel(file_path)
        df = pd.concat([df, one], ignore_index=True)
        print(f'已并入 {file}：当前共 {len(df)} 行')

print('\n=== 合并完成 ===')
print('总行数：', len(df))
print(df)


> **提示**：`sorted()` 这一步很关键。
> `os.listdir()` 返回的顺序**由操作系统决定，不保证稳定**。
> 不排序的话，每次运行结果的行顺序可能都不一样，排查问题时很痛苦。

> 另外补充两点关于**性能**的：上面的写法每读一个文件就 `concat` 一次，
> 文件一多（比如几百个）会**越来越慢**（每次都复制整张表）。
> 更高效的做法是先收集到列表，最后**一次性** `concat`：


In [ ]:
# 【更推荐】先收集，最后一次性合并 —— 文件多的时候快得多
frames = []
for file in sorted(os.listdir(INPUT_DIR)):
    if not file.endswith('.xlsx'):
        continue
    one = pd.read_excel(os.path.join(INPUT_DIR, file))
    frames.append(one)

df2 = pd.concat(frames, ignore_index=True)
print('一次性 concat 的结果：', df2.shape)
print('和逐个 concat 的结果一样吗？', df.shape == df2.shape)


---
# 三、加一列「来源标识」——回溯数据来源

合并之后就"分不清谁是谁"了：一行数据到底是哪个地区来的？
加一列来源标识，以后出问题能顺着查回去。

原稿的做法是**取文件名的前 3 个字符**：

```python
identifier = file_name_without_ext[:3]
```

这在小场景能凑合，但很脆弱 —— 文件名一改（比如叫 `2024华东.xlsx`），
标识就全错了。**直接用完整文件名最稳**（真嫌长再回头简化）。


In [ ]:
frames = []
for file in sorted(os.listdir(INPUT_DIR)):
    if not file.endswith('.xlsx'):
        continue
    one = pd.read_excel(os.path.join(INPUT_DIR, file))

    # 【推荐】用完整文件名当来源标识，绝对不会有歧义
    one['来源文件'] = file

    # 如果确实只想要短名字，用 os.path.splitext 去掉扩展名，比切片安全
    one['地区'] = os.path.splitext(file)[0]

    frames.append(one)

df_src = pd.concat(frames, ignore_index=True)
print('带来源标识的合并结果：')
print(df_src)


In [ ]:
print('按地区统计：')
summary = (df_src.assign(销售额=df_src['数量'] * df_src['单价'])
                 .groupby('地区', as_index=False)
                 .agg(记录数=('商品', 'size'), 销售额=('销售额', 'sum'))
                 .sort_values('销售额', ascending=False))
print(summary)


---
# 四、把结果写回去 —— 这里有个大坑

接下来要把合并结果存成 Excel。
**坑就在"存到哪里"这个看似不起眼的决定上。**


## 4.1 复现：结果写进输入目录，数据会翻倍

原稿是这么写的（把结果写回**正在被扫描的那个目录**）：

```python
path = r'C:\Users\leisuredzz\Desktop\杂物库\全国销售数据'
...
df.to_excel(path + r'\全国销售数据.xlsx', index=False)   # 写回同一个目录
```

原稿用 `if file == '全国销售数据.xlsx': continue` 把它跳到，
所以侥幸没出事。但**这个防护很脆弱**：文件被改个名，就失效了。

我们现在**故意复现**一下这个坑：


In [ ]:
# 第 1 次运行：把结果写进 INPUT_DIR 里面
result_path = os.path.join(INPUT_DIR, '全国销售数据.xlsx')
df_src.to_excel(result_path, index=False)

files_first = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith('.xlsx'))
print('第一次运行后的目录：')
for f in files_first:
    print('  ', f)

# 再来一次"合并" —— 注意这次目录里多了一个总表
rows_first = 0
for f in files_first:
    rows_first += len(pd.read_excel(os.path.join(INPUT_DIR, f)))
print(f'\n目录里所有 Excel 加起来共 {rows_first} 行（原始数据只有 {len(df_src)} 行）')


In [ ]:
# 第 2 次运行（模拟下次又跑一遍这个脚本）
frames2 = []
for f in sorted(os.listdir(INPUT_DIR)):
    if f.endswith('.xlsx'):
        frames2.append(pd.read_excel(os.path.join(INPUT_DIR, f)))
df_bad = pd.concat(frames2, ignore_index=True)

print(f'第 2 次运行后：{len(df_bad)} 行   （第 1 次是 {len(df_src)} 行）')
print(f'-> 数据翻倍了！这就是"结果文件被当成输入"的后果。')
print(f'-> 再跑第 3 次就是 {len(df_bad) * 2} 行，越滚越大。')


## 4.2 正确做法：输入和输出**分开放**

一眼就看出问题：**输入目录和输出目录绝对不能是同一个。**

顺便说，原稿还有个更大的隐患 —— 那个写死的绝对路径：

```python
path = r'C:\Users\leisuredzz\Desktop\杂物库\全国销售数据'
```

换一台电脑、换一个用户名、把文件夹挪个位置，代码立刻报 `FileNotFoundError`。
**代码里不要出现自己电脑的绝对路径**，要么用相对路径，
要么把路径做成变量放在开头（像我们这样）。


In [ ]:
# 现在要把误写进去的结果文件"请出去"，恢复干净的输入目录。
#
# 注意这里用的是 **move（移动）而不是 remove（删除）**：
# 移动只是换个位置，**不丢数据**，也不受"删除权限 / 安全策略"影响。
# 这是处理"放错位置的文件"更稳的做法。
misplaced = os.path.join(MERGED_DIR, '误写在输入目录的总表.xlsx')
os.replace(result_path, misplaced)      # 移动同名文件时会直接覆盖，重复运行安全
print('已把误写的总表移到：', misplaced)
print('输入目录现在只剩原始报表：', sorted(os.listdir(INPUT_DIR)))


In [ ]:
# 【正确做法】输出到独立的目录

def merge_all_files(input_dir, pattern='*.xlsx', source_col='来源文件'):
    """把一个目录下所有匹配的文件合并成一张表。

    参数
    ----
    input_dir  : 待扫描的目录
    pattern    : 文件名通配符，如 '*.xlsx'
    source_col : 用来记录来源文件名的列名；传 None 就不加

    返回
    ----
    合并后的 DataFrame
    """
    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not paths:
        raise FileNotFoundError(f'在 {input_dir} 里没找到匹配 {pattern} 的文件')

    frames = []
    for p in paths:
        one = pd.read_excel(p)
        if source_col:
            one[source_col] = os.path.basename(p)     # 只留文件名，不留整条路径
        frames.append(one)

    return pd.concat(frames, ignore_index=True)


merged = merge_all_files(INPUT_DIR)
print('合并完成：', merged.shape)
print(merged.head())


In [ ]:
# 写到【另一个】目录，这样反复运行都不会污染输入
out_path = os.path.join(MERGED_DIR, '全国销售数据_合并结果.xlsx')
merged.to_excel(out_path, index=False)

print('已输出到：', out_path)
print()
print('输入目录（干净）：', sorted(os.listdir(INPUT_DIR)))
print('输出目录：        ', sorted(os.listdir(MERGED_DIR)))


In [ ]:
# 反复运行 3 次，验证结果行数稳定不翻倍
for i in range(3):
    again = merge_all_files(INPUT_DIR)
    again.to_excel(out_path, index=False)
    print(f'第 {i + 1} 次重跑 -> {len(again)} 行')

print('\n行数始终', len(merged), '，说明这个写法是【幂等】的：')
print('  重复运行不会改变结果，这是一份合格的数据处理脚本的底线。')


---
# 五、更稳一些的写法

生产中光"能跑"不够，还要"跑得放心"。几个可以加上的保险：


In [ ]:
# 保险 1：写入前先校验"各文件的列是否一致"
def check_columns(input_dir, pattern='*.xlsx'):
    """检查目录里各文件的列名是否一致，返回不一致的文件清单。"""
    paths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    base_cols = None
    problems = []
    for p in paths:
        cols = list(pd.read_excel(p, nrows=0).columns)   # nrows=0 只读表头，很快
        if base_cols is None:
            base_cols = cols
        elif cols != base_cols:
            problems.append((os.path.basename(p), cols))
    return base_cols, problems


base, problems = check_columns(INPUT_DIR)
print('基准列：', base)
print('列不一致的文件：', problems if problems else '无 ✅')


In [ ]:
# 演示：故意造一个"列不一样"的文件，看看能不能查出来
weird = pd.DataFrame({'日期': ['2024-02-01'], '商品名': ['电视'], '销量': [3]})
weird_path = os.path.join(INPUT_DIR, '异常的西北.xlsx')
weird.to_excel(weird_path, index=False)

base, problems = check_columns(INPUT_DIR)
print('列不一致的文件：')
for name, cols in problems:
    print(f'  {name} -> {cols}')


In [ ]:
# 如果不管列名直接合并会怎样？—— 会多出 NaN 列，而且信息错位
naive = pd.concat([pd.read_excel(os.path.join(INPUT_DIR, f))
                   for f in sorted(os.listdir(INPUT_DIR)) if f.endswith('.xlsx')],
                  ignore_index=True)
print('直接合并（不校验）的结果：')
print(naive.tail(3))
print('\n总行数：', len(naive), '（4 个正常文件 19 行 + 异常文件 1 行）')
print('列变成：', list(naive.columns))
print('-> 商品名、销量这两列只对"西北"那行有值，其余全是 NaN；')
print('   而"商品""数量"两列反过来。行数看着对得上，其实是【错位】的。')
print('-> 所以合并前一定要先校验各文件列名是否一致。')


In [ ]:
# 把这个异常文件也**移走**（同样用 move 而不是 remove）
os.replace(weird_path, os.path.join(MERGED_DIR, '异常的西北.xlsx'))
print('已移走异常文件，目录恢复：', sorted(os.listdir(INPUT_DIR)))


---
# 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 输出写进输入目录 | **数据翻倍**，越跑越大 | 输入输出分开；或用 `if` 排除结果文件 |
| 写死绝对路径 | 换电脑就 `FileNotFoundError` | 用相对路径或变量；`os.path.basename` 取名字 |
| 不排序直接遍历 | 行顺序每次都可能不同 | `sorted(...)` 或 `glob.glob` + `sorted` |
| 每读一个就 `concat` | 文件多时明显变慢 | 先收集列表，最后一次 `concat` |
| 忘了 `ignore_index=True` | 行号是 0,1,2,0,1,2... 重复 | 加上它 |
| 各文件列不一致 | 静默错位 / 一堆 NaN | 先 `check_columns` 校验 |
| `.xls` 读不了 | `ImportError: Missing optional dependency 'xlrd'` | 装 `xlrd`，或让来源统一成 `.xlsx` |
| 文件名当标识用切片取前 3 字 | 文件一改名标识就错 | 用 `os.path.splitext` 取完整名 |
| 目录里混着 `~$xxx.xlsx` | 读到 Excel 临时文件报错 | 过滤掉 `~$` 开头的文件 |
| 空目录 | `concat` 空列表报错 | 先判断文件列表是否为空 |
| 用"删文件"来保证可重复运行 | 依赖删除权限，某些环境会被拦下 | **能用"覆盖"就别用"删除"**；放错位置用 `os.replace` 移走 |


In [ ]:
# 最后两条坑顺手演示一下

# ① 过滤 Excel 的临时文件（打开 Excel 时会出现 ~$ 开头的隐藏文件）
all_files = ['华东.xlsx', '~$华东.xlsx', '华北.xlsx']
clean = [f for f in all_files if not f.startswith('~$')]
print('过滤临时文件：', all_files)
print('           ->', clean)

# ② 空列表 concat 会报错
print()
try:
    pd.concat([])
except ValueError as e:
    print('空列表 concat 报错：', e)
    print('-> 所以脚本开头要先判断：if not paths: raise FileNotFoundError(...)')


---
# 动手练习

1. **多 sheet 合并**：让每个地区的 Excel 里有"线上"和"线下"两个 sheet，
   把它们全部合并成一张表，并加一列标记是哪个渠道。
2. **CSV 版**：把上面流程改成读 `.csv`（注意 `encoding='utf-8-sig'`），
   并处理"有的文件是 GBK 编码"的情况。
3. **加时间维度**：合并后按月汇总销售额，画一张趋势表。
4. **健壮性**：给 `merge_all_files` 加上"遇到坏文件跳过并记录"的能力，
   返回 `(合并结果, 出问题的文件清单)`。
5. **反向操作**：把一张大表按"地区"列**拆成多个 Excel**，每个地区一个文件。
   提示：`groupby` + `to_excel`。

---
# 答案思路

**1. 多 sheet 合并**

```python
frames = []
for f in sorted(os.listdir(INPUT_DIR)):
    if not f.endswith('.xlsx'):
        continue
    sheets = pd.read_excel(os.path.join(INPUT_DIR, f), sheet_name=None)  # None = 读全部 sheet
    for sheet_name, one in sheets.items():
        one['渠道'] = sheet_name
        one['来源文件'] = f
        frames.append(one)
df_all = pd.concat(frames, ignore_index=True)
```

**2. CSV 版（含编码兜底）**

```python
def read_any_csv(path):
    for enc in ('utf-8-sig', 'gbk', 'utf-8'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'{path} 编码识别失败')
```

**3. 按月汇总**

```python
trend = (merged.assign(月=pd.to_datetime(merged['日期']).dt.to_period('M').astype(str),
                       销售额=merged['数量'] * merged['单价'])
               .groupby('月', as_index=False)['销售额'].sum())
print(trend)
```

**4. 健壮版**

```python
def merge_all_files_safe(input_dir, pattern='*.xlsx'):
    paths = sorted(p for p in glob.glob(os.path.join(input_dir, pattern))
                   if not os.path.basename(p).startswith('~$'))
    if not paths:
        raise FileNotFoundError(f'{input_dir} 下没有匹配 {pattern} 的文件')
    frames, bad = [], []
    for p in paths:
        try:
            one = pd.read_excel(p)
            one['来源文件'] = os.path.basename(p)
            frames.append(one)
        except Exception as e:
            bad.append((os.path.basename(p), f'{type(e).__name__}: {e}'))
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), bad
```

**5. 反向拆分**

```python
for region, group in merged.groupby('地区'):
    group.to_excel(os.path.join(MERGED_DIR, f'{region}_拆分.xlsx'), index=False)
    print(f'{region}: {len(group)} 行')
```
